# 🏍️ Helmet Violation Detection + Plate OCRRuns the full pipeline on **Google Colab**:1. Download the Kaggle dataset2. Train YOLOv83. Detect riders without helmets and read their number plates**Before you start:** enable GPU — `Runtime → Change runtime type → T4 GPU`.

## 1. Install packages

In [ ]:
!pip install -q ultralytics kagglehub easyocr pyyaml

## 2. Kaggle credentialsGo to kaggle.com → your profile → Settings → **Create New Token**. That downloads `kaggle.json`. Upload it below.

In [ ]:
from google.colab import filesimport os, shutiluploaded = files.upload()  # pick your kaggle.json hereos.makedirs('/root/.config/kaggle', exist_ok=True)shutil.move('kaggle.json', '/root/.config/kaggle/kaggle.json')os.chmod('/root/.config/kaggle/kaggle.json', 0o600)print('Kaggle credentials set.')

## 3. Download the dataset

In [ ]:
import kagglehubdataset_path = kagglehub.dataset_download(    "aneesarom/rider-with-helmet-without-helmet-number-plate")print("Downloaded to:", dataset_path)

## 4. Peek at what's inside

In [ ]:
import osfor root, dirs, fs in os.walk(dataset_path):    depth = root.replace(dataset_path, '').count(os.sep)    if depth > 2:        continue    indent = '  ' * depth    print(f'{indent}{os.path.basename(root)}/')    for f in fs[:3]:        print(f'{indent}  {f}')    if len(fs) > 3:        print(f'{indent}  ... (+{len(fs)-3} more)')

## 5. Fix `data.yaml`The uploader's YAML usually has absolute paths from their own machine. We rewrite it to point at the folders we actually have.

In [ ]:
import glob, yamlfrom pathlib import Path# find the yaml file that came with the datasetyaml_files = glob.glob(f"{dataset_path}/**/*.yaml", recursive=True)original_yaml = yaml_files[0]print("Original yaml:", original_yaml)with open(original_yaml) as f:    data = yaml.safe_load(f)print("\nOriginal contents:")print(data)

In [ ]:
root = Path(dataset_path)def find_split(*names):    for name in names:        for pattern in (f"**/{name}/images", f"**/images/{name}"):            for hit in root.glob(pattern):                if hit.is_dir() and any(hit.iterdir()):                    return hit    return Nonetrain_dir = find_split("train")val_dir   = find_split("valid", "val")test_dir  = find_split("test")print("train:", train_dir)print("val:  ", val_dir)print("test: ", test_dir)data["path"]  = str(root)data["train"] = str(train_dir.relative_to(root))data["val"]   = str(val_dir.relative_to(root))if test_dir:    data["test"] = str(test_dir.relative_to(root))# ultralytics prefers names as a dictif isinstance(data.get("names"), list):    data["names"] = {i: n for i, n in enumerate(data["names"])}with open("data.yaml", "w") as f:    yaml.safe_dump(data, f, sort_keys=False)print("\nFinal data.yaml:")print(open("data.yaml").read())

## 6. Look at a few training images

In [ ]:
import cv2, matplotlib.pyplot as pltsamples = sorted(train_dir.glob("*"))[:6]fig, axes = plt.subplots(2, 3, figsize=(15, 8))for ax, path in zip(axes.flat, samples):    img = cv2.imread(str(path))    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))    ax.axis("off")    ax.set_title(path.name, fontsize=9)plt.tight_layout()plt.show()

## 7. Train YOLOv8Small model + 30 epochs — finishes in ~15–25 min on a T4. Bump `epochs=100` and use `yolov8s.pt` for better accuracy.

In [ ]:
from ultralytics import YOLOmodel = YOLO("yolov8n.pt")   # try yolov8s.pt or yolov8m.pt for more accuracymodel.train(    data="data.yaml",    epochs=30,    imgsz=640,    batch=16,    project="runs/helmet",    name="train",    patience=10,)

## 8. Training curves

In [ ]:
from IPython.display import ImageImage("runs/helmet/train/results.png")

## 9. Validate on the val set

In [ ]:
metrics = model.val()print(f"mAP50:    {metrics.box.map50:.3f}")print(f"mAP50-95: {metrics.box.map:.3f}")

## 10. Quick predictions on test images

In [ ]:
from google.colab.patches import cv2_imshow# grab a handful of test (or val) imagesif test_dir:    test_imgs = sorted(test_dir.glob("*"))[:5]else:    test_imgs = sorted(val_dir.glob("*"))[:5]for img_path in test_imgs:    result = model.predict(str(img_path), conf=0.35, verbose=False)[0]    annotated = result.plot()    print(f"\n{img_path.name}")    cv2_imshow(annotated)

## 11. Full pipeline: detect violations + read platesLogic:- If a rider box contains a *without-helmet* head → **violation**.- Crop the number plate inside that rider's box and run OCR.

In [ ]:
import easyocr# loads once; first run downloads ~64 MBreader = easyocr.Reader(["en"], gpu=True)

In [ ]:
# figure out which class id means what — dataset naming variesnames = model.namesprint("Model classes:", names)def find_id(*aliases):    aliases = {a.lower() for a in aliases}    for i, n in names.items():        clean = n.lower().replace("_", " ").strip()        if clean in aliases:            return i    return NoneRIDER     = find_id("rider", "motorcyclist")HELMET    = find_id("with helmet", "helmet")NO_HELMET = find_id("without helmet", "no helmet", "wo helmet")PLATE     = find_id("number plate", "plate", "license plate")print(f"rider={RIDER}  helmet={HELMET}  no_helmet={NO_HELMET}  plate={PLATE}")assert RIDER is not None and NO_HELMET is not None, (    "Class not resolved — add the exact name from `Model classes` above "    "into find_id() aliases.")

In [ ]:
import numpy as npdef center_in(outer, inner):    cx = (inner[0] + inner[2]) / 2    cy = (inner[1] + inner[3]) / 2    return outer[0] <= cx <= outer[2] and outer[1] <= cy <= outer[3]def read_plate(crop):    if crop.size == 0:        return ""    # upscale tiny crops so OCR has something to work with    h = crop.shape[0]    if h < 60:        s = 60 / h        crop = cv2.resize(crop, None, fx=s, fy=s, interpolation=cv2.INTER_CUBIC)    texts = reader.readtext(crop, detail=0)    return "".join(ch for ch in " ".join(texts).upper() if ch.isalnum() or ch == " ")def detect_violations(image_path):    img = cv2.imread(str(image_path))    res = model.predict(img, conf=0.35, verbose=False)[0]    boxes = res.boxes.xyxy.cpu().numpy()    clses = res.boxes.cls.cpu().numpy().astype(int)    riders     = [b for b, c in zip(boxes, clses) if c == RIDER]    no_helmets = [b for b, c in zip(boxes, clses) if c == NO_HELMET]    plates     = [b for b, c in zip(boxes, clses) if c == PLATE]    annotated = res.plot()    violations = []    for r in riders:        # is there a no-helmet head inside this rider box?        if not any(center_in(r, nh) for nh in no_helmets):            continue        # yes — find its plate        rider_plates = [p for p in plates if center_in(r, p)]        plate_text = ""        if rider_plates:            # take the biggest plate            p = max(rider_plates, key=lambda b: (b[2]-b[0]) * (b[3]-b[1]))            x1, y1, x2, y2 = map(int, p)            crop = img[max(0,y1):y2, max(0,x1):x2]            plate_text = read_plate(crop)            # write the OCR text over the plate            cv2.putText(annotated, plate_text or "?",                        (x1, max(20, y1 - 8)),                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)        violations.append(plate_text)    return annotated, violations

## 12. Run the pipeline

In [ ]:
for img_path in test_imgs:    annotated, viols = detect_violations(img_path)    print(f"\n{img_path.name}  →  {len(viols)} violation(s): {viols}")    cv2_imshow(annotated)

## 13. Try your own imageUpload any traffic photo (JPG/PNG) and see what the model finds.

In [ ]:
from google.colab import files as _filesmy_upload = _files.upload()for name in my_upload:    annotated, viols = detect_violations(name)    print(f"\n{name}  →  {len(viols)} violation(s): {viols}")    cv2_imshow(annotated)

## 14. Save the trained modelDownload `best.pt` so you can reuse it without retraining.

In [ ]:
from google.colab import files as _files_files.download("runs/helmet/train/weights/best.pt")

---### Tips- **Class name mismatch?** Cell 11 prints the model's class names. Add the exact strings to the `find_id(...)` aliases if a role comes out as `None`.- **Better accuracy:** train longer (`epochs=100`) and use `yolov8s.pt` or `yolov8m.pt`.- **Better OCR:** swap EasyOCR for PaddleOCR (`!pip install paddleocr paddlepaddle` and use `PaddleOCR(use_angle_cls=True, lang='en')`) — it's usually noticeably better on angled plates.- **Video input:** loop with `cv2.VideoCapture`, call `detect_violations` per frame, write to a `cv2.VideoWriter`.